In [ ]:
import share_modules
import const

test_result_path = "../result/english/suffix/test-t5-base-e0-extract-tuple-constrained-model-True-suffix-lr-5e4-1ep (3).txt"
test_sents, truth_csi, test_truths = share_modules.read_english_raw_file("../data/raw_data/english/test.txt")
_, test_predicts = share_modules.read_data_file(test_result_path)


In [273]:
# my_dict = {"a": 1, "b": 2, "c": 3, "d": 4}
# target_key = "b"

def get_next_pos(my_dict, target_key):
  next_key = None
  for key, _ in my_dict.items():
    if key == target_key:
      next_key = list(my_dict.keys())[list(my_dict.values()).index(my_dict[target_key]) + 1]
      break  # Exit loop after finding the next key

  return next_key

# print(get_next_pos(my_dict=my_dict, target_key=target_key))
def reorder_lable_tuple(text):
  
    order = ["[S]", "[O]", "[A]", "[P]", "[L]"]

  # Split the string into words
    words = text.split()[1:-1]
    try:
        keyword_positions = {_o: words.index(_o) for _o in order}
    except ValueError:
        return ""
    # print(keyword_positions)

    sorted_pos = dict(sorted(keyword_positions.items(), key=lambda item: item[1]))

    def get_next_pos(my_dict, target_key):
        next_key = None
        for key, _ in my_dict.items():
            if key == target_key:
                next_key = list(my_dict.keys())[list(my_dict.values()).index(my_dict[target_key]) + 1]
                break  # Exit loop after finding the next key

        return next_key

    new_string = ""
    for _o in order[:-1]:
        if _o in words:
            next_tag = get_next_pos(sorted_pos, _o)
            new_string = new_string + " ".join(words[keyword_positions[_o]: sorted_pos[next_tag]]) + " "
        else:
            break
    
    if "[L]" in words:
        new_string =  new_string + " ".join(words[keyword_positions["[L]"]:])

    return new_string
# Example usage
# text = "[P] better [A] [UNK] [O] that [S] Fuji film camera [L] Better"
# reordered_text = reorder_lable_tuple(text)
# print(reordered_text)

error_predicts = []
reorder_test_predicts = []
for predict in test_predicts:
    pred_list = predict.split(";")
    temp = []
    for pred in pred_list:
        reorder_pred = reorder_lable_tuple(pred)
        if reorder_pred != "":
            temp.append(f"({reorder_pred})")
    reorder_string = ";".join(temp)
    reorder_test_predicts.append(reorder_string)
    

# reorder_test_predicts

In [274]:
from sklearn.metrics import classification_report
def predict_csi_result(test_predicts, truth_csi):
    test_csi = []
    for pred in test_predicts:
        if pred != "( [S] [UNK] [O] [UNK] [A] [UNK] [P] [UNK] [L] [UNK] )":
            test_csi.append(1)
        else:
            test_csi.append(0)
    report = classification_report(truth_csi, test_csi, output_dict=True)
    print(f"CSI classification report: \n {report}")
    return test_csi

test_csi = predict_csi_result(test_predicts, truth_csi)
    

CSI classification report: 
 {'0': {'precision': 0.9566929133858267, 'recall': 0.759375, 'f1-score': 0.8466898954703833, 'support': 320.0}, '1': {'precision': 0.8108108108108109, 'recall': 0.967741935483871, 'f1-score': 0.8823529411764706, 'support': 341.0}, 'accuracy': 0.8668683812405447, 'macro avg': {'precision': 0.8837518620983188, 'recall': 0.8635584677419355, 'f1-score': 0.8645214183234269, 'support': 661.0}, 'weighted avg': {'precision': 0.8814345215884282, 'recall': 0.8668683812405447, 'f1-score': 0.8650879266137657, 'support': 661.0}}


In [275]:
import pandas as pd

test_df = pd.DataFrame({"sentence": test_sents, "predict": test_predicts, "truth": test_truths, "test_csi": test_csi,  "truth_csi": truth_csi})
test_df

,sentence,predict,truth,test_csi,truth_csi
0,"My 10 year old Fuji film camera , which was a ...",( [S] that [A] [UNK] [S] 10 year old Fuji film...,[[[5&&Fuji 6&&film 7&&camera];[20&&that];[];[1...,1,1
1,After taking close to 300 and the battery mete...,( [S] [UNK] [O] [UNK] [A] [UNK] [P] [UNK] [L] ...,[[[];[];[];[];[]]],0,0
2,As for comparisions with other Canon Powershot...,( [S] Powershot SD630 [P] larger [O] Canon Pow...,[[[5&&other 6&&Canon 7&&Powershot 8&&cameras];...,1,1
3,Viewfinder/LCD The LCD gives you 99-100 % fram...,( [S] LCD [A] framing [P] gives you 99-100 % f...,[[[];[];[];[];[]]],1,0
4,Outdoors the clarity is outstanding .,( [S] [UNK] [O] [UNK] [A] clarity [P] outstand...,[[[];[];[];[];[]]],1,0
...,...,...,...,...,...
656,The integrated flash can be used to trigger re...,( [S] [UNK] [O] [UNK] [A] [UNK] [P] [UNK] [L] ...,[[[];[];[];[];[]]],0,0
657,I get better results off my much older Sony Cy...,( [S] Sony Cybershot DSP-70 [A] results [P] be...,[[[];[9&&Sony 10&&Cybershot 11&&DSP-70];[4&&re...,1,1
658,I recently gave up my Sony F-707 for this DSLR...,( [S] Sony F-707 [A] [UNK] [P] gave up [S] Son...,[[[];[];[];[];[]]],1,0
659,I use it right now for sport games and it work...,( [S] [UNK] [O] [UNK] [A] [UNK] [P] [UNK] [L] ...,[[[];[];[];[];[]]],0,0


# Extract Predict Position

In [276]:
import re

def extract_elements(input_string):
    input_list = input_string.split(';')
    pattern = re.compile(r"\[([SOAPL])\] (.*?)(?=\s\[|\Z)")
    result=[]
    for i in input_list:
        i = i.strip()
        match = re.findall(pattern, i[1:-1].strip())
        if match:
            result.append(match)
    return result



elem_dict = ['subject', 'object', 'aspect', 'predicate', 'label']
all_labels, all_predictions, error_preds = [], [], []
for index in range(len(test_sents)):
    predict_list = extract_elements(test_predicts[index])

    print(predict_list)
    if len(predict_list)==0:
        error_preds.append(f"{test_sents[index]} ===> {test_predicts[index]}")
    else:
        all_predictions.append(predict_list)
    

            
print(f"The number of error predictions: {len(error_preds)}")


[[('S', 'that'), ('A', '[UNK]'), ('S', '10 year old Fuji film camera'), ('P', 'better'), ('L', 'Better')]]
[[('S', '[UNK]'), ('O', '[UNK]'), ('A', '[UNK]'), ('P', '[UNK]'), ('L', '[UNK]')]]
[[('S', 'Powershot SD630'), ('P', 'larger'), ('O', 'Canon Powershot cameras'), ('A', 'LCD monitor'), ('L', 'Better')]]
[[('S', 'LCD'), ('A', 'framing'), ('P', 'gives you 99-100 % framing'), ('O', 'optical viewfinder'), ('L', 'Better')], [('S', 'LCD'), ('A', '[UNK]'), ('P', 'gives you 85 % framing'), ('O', 'optical viewfinder'), ('L', 'Better')]]
[[('S', '[UNK]'), ('O', '[UNK]'), ('A', 'clarity'), ('P', 'outstanding'), ('L', 'Better')]]
[[('S', '[UNK]'), ('O', '[UNK]'), ('A', '[UNK]'), ('P', '[UNK]'), ('L', '[UNK]')]]
[[('S', 'digital Elph line'), ('A', 'manual controls'), ('P', 'Like'), ('S', '[UNK]'), ('L', 'Equal')]]
[[('S', '[UNK]'), ('O', '[UNK]'), ('A', '[UNK]'), ('P', '[UNK]'), ('L', '[UNK]')]]
[[('S', '[UNK]'), ('O', '[UNK]'), ('A', '[UNK]'), ('P', '[UNK]'), ('L', '[UNK]')]]
[[('S', '[UNK]'),

In [277]:
def convert_to_dict(data):
  semantic_roles = {
      'S': 'subject',
      'O': 'object',
      'A': 'aspect',
      'P': 'predicate',
      'L': 'label'
  }

  result = {}
  for role, element in data:
    if role == 'L' and element == '[UNK]':      
      result[semantic_roles[role]] = ''
    elif element =='[UNK]':
      result[semantic_roles[role]] = []
    elif role == 'L':
      result[semantic_roles[role]] = element
    else:
      temp = element.split(' ')
      new_temp = []
      for ele in temp:
          if "'s" in ele:
            new_temp.extend([ele[:-2], "'s"])
          elif "n't" in ele:
            new_temp.extend([ele[:-3], "n't"])
          else:
            new_temp.append(ele)

      result[semantic_roles[role]] = new_temp

  return result

label_tuples = []
def init_tuple(elem_dict):
    dict = {}
    for key in elem_dict:
        if key != 'label':
            dict[key] = []
        else:
            dict[key] = ''
    return dict

error_quintuple_cnt = 0
for predict in all_predictions:
    new_labels = []
    for tup in predict:
        # quintupe = init_tuple(elem_dict)
        quintuple = convert_to_dict(tup)
        if len(quintuple) != 5:
            error_quintuple_cnt = error_quintuple_cnt + 1
            print(f"error prediction: {quintuple}")
        else:
            new_labels.append(quintuple)
        # print(quintuple)

    label_tuples.append(new_labels)
#         for index, q in enumerate(tup):
#             if elem_dict[index] != 'label':
#                 if tup[index] == '[UNK]':
#                     quintupe[elem_dict[index]] = []
#                 else:
#                     quintupe[elem_dict[index]].extend(tup[index].split(' '))
#             else:
#                 if tup[index] == '[UNK]':
#                     quintupe[elem_dict[index]] = ''
#                 else:
#                     quintupe[elem_dict[index]] = tup[index]


#         new_labels.append(quintupe)
    
#     label_tuples.append(new_labels)

print(len(label_tuples))
print("Number of error quintuple: ", error_quintuple_cnt)



error prediction: {'subject': ['10', 'year', 'old', 'Fuji', 'film', 'camera'], 'aspect': [], 'predicate': ['better'], 'label': 'Better'}
error prediction: {'subject': [], 'aspect': ['manual', 'controls'], 'predicate': ['Like'], 'label': 'Equal'}
error prediction: {'subject': ['Nikon', 'Coolpix', 'L5'], 'predicate': ['half'], 'aspect': ['price'], 'label': 'Better'}
error prediction: {'subject': ['It'], 'predicate': ['bridges'], 'aspect': ['gap'], 'label': 'Better'}
error prediction: {'subject': ['top', 'of', 'the', 'camera'], 'predicate': ['different'], 'aspect': [], 'label': 'Better'}
error prediction: {'subject': [], 'aspect': ['true', 'night', 'pic'], 'predicate': ['better'], 'label': 'Better'}
error prediction: {'subject': [], 'aspect': ['price'], 'predicate': ['not', 'find', 'a', 'better'], 'label': 'Worse'}
error prediction: {'subject': ['40D'], 'aspect': ['optical', 'sensor'], 'predicate': ['larger'], 'label': 'Better'}
error prediction: {'subject': ['higher', '16', 'megapixel', 

In [278]:
null_label = init_tuple(elem_dict=elem_dict)
cnt = 0
for tup in label_tuples:
    if tup != [null_label]:
        cnt +=1
        print(tup)
print(f"Number of extracted quintuple label: {cnt}")

[]
[{'subject': ['Powershot', 'SD630'], 'predicate': ['larger'], 'object': ['Canon', 'Powershot', 'cameras'], 'aspect': ['LCD', 'monitor'], 'label': 'Better'}]
[{'subject': ['LCD'], 'aspect': ['framing'], 'predicate': ['gives', 'you', '99-100', '%', 'framing'], 'object': ['optical', 'viewfinder'], 'label': 'Better'}, {'subject': ['LCD'], 'aspect': [], 'predicate': ['gives', 'you', '85', '%', 'framing'], 'object': ['optical', 'viewfinder'], 'label': 'Better'}]
[{'subject': [], 'object': [], 'aspect': ['clarity'], 'predicate': ['outstanding'], 'label': 'Better'}]
[]
[{'subject': ['both', 'Canons'], 'predicate': ['better'], 'aspect': ['AF'], 'object': [], 'label': 'Better'}, {'subject': ['both', 'Canons'], 'predicate': ['better'], 'aspect': ['white', 'balance', 'adjustments'], 'object': [], 'label': 'Better'}, {'subject': ['both', 'Canons'], 'predicate': ['lower'], 'aspect': ['ISO', 'noise'], 'object': [], 'label': 'Better'}, {'subject': ['both', 'Canons'], 'predicate': ['sharper'], 'aspe

In [279]:
s1 = u'ÀÁÂÃÈÉÊÌÍÒÓÔÕÙÚÝàáâãèéêìíòóôõùúýĂăĐđĨĩŨũƠơƯưẠạẢảẤấẦầẨẩẪẫẬậẮắẰằẲẳẴẵẶặẸẹẺẻẼẽẾếỀềỂểỄễỆệỈỉỊịỌọỎỏỐốỒồỔổỖỗỘộỚớỜờỞởỠỡỢợỤụỦủỨứỪừỬửỮữỰựỲỳỴỵỶỷỸỹ'
s0 = u'AAAAEEEIIOOOOUUYaaaaeeeiioooouuyAaDdIiUuOoUuAaAaAaAaAaAaAaAaAaAaAaAaEeEeEeEeEeEeEeEeIiIiOoOoOoOoOoOoOoOoOoOoOoOoUuUuUuUuUuUuUuYyYyYyYy'
def remove_accents(input_str):
	s = ''
	for c in input_str:
		if c in s1:
			s += s0[s1.index(c)]
		else:
			s += c
	return s

def check_sublist_in_list(test_list, sub):
    
    if len(sub) > 1:
        for start in range(len(sub)-1):
            sublist = sub[start:]
            for idx in range(len(test_list) - len(sublist) + 1):
                if test_list[idx: idx + len(sublist)] == sublist:
                    return len(sub) - start, idx
    else:
        for idx in range(len(test_list) - len(sub) + 1):
                if test_list[idx: idx + len(sub)] == sub:
                    return len(sub), idx
                
    if len(sub) > 1:
        for start in range(len(sub)):
            sublist = sub[:len(sub)-start]
            for idx in range(len(test_list) - len(sublist) + 1):
                if test_list[idx: idx + len(sublist)] == sublist:
                    return len(sub) - start, idx
    else:
        for idx in range(len(test_list) - len(sub) + 1):
                if test_list[idx: idx + len(sub)] == sub:
                    return len(sub), idx
            
    return -1, -1

check_sublist_in_list([1105, 320, 5, 774, 1193, 632, 54, 5365], [320, 5, 774])

(3, 1)

In [280]:
cnt = 0
for index, label in enumerate(label_tuples):
    # gold_label = remove_accents(test_sents[index])
    for tup in label:
        for key in elem_dict:
            if tup[key] and key !='label':
                new_tup = " ".join(tup[key])
                # new_tup = remove_accents(new_tup)

                if test_sents[index].find(new_tup) == -1:
                    cnt += 1
                    print(test_sents[index])
                    print(tup)
                    print(tup[key])

print(f"Number of not finding element: {cnt}")

Viewfinder/LCD The LCD gives you 99-100 % framing while the optical viewfinder gives you about 85 % framing . 
{'subject': ['LCD'], 'aspect': [], 'predicate': ['gives', 'you', '85', '%', 'framing'], 'object': ['optical', 'viewfinder'], 'label': 'Better'}
['gives', 'you', '85', '%', 'framing']
The A630 has 4 AA 's , not 2 like the 710 and 570 . 
{'subject': ['A630'], 'predicate': ['has', '4', "AA'", "'s", 'not', '2', 'like'], 'object': ['710'], 'aspect': ['AA'], 'label': 'Worse'}
['has', '4', "AA'", "'s", 'not', '2', 'like']
No , the sensors are not as , uh , sensitive as some of the other cameras on the market , and there may be fewer of them , too . 
{'subject': ['sensors'], 'object': ['other', 'cameras'], 'predicate': ['not', 'as,', 'uh,', 'sensitive'], 'aspect': [], 'label': 'Worse'}
['not', 'as,', 'uh,', 'sensitive']
No , the sensors are not as , uh , sensitive as some of the other cameras on the market , and there may be fewer of them , too . 
{'subject': ['sensors'], 'object': ['

In [281]:
from transformers import AutoTokenizer
gold_token_col, model_token_col, gold_id_col = [],  [], []

tokenizer = AutoTokenizer.from_pretrained('google-t5/t5-base')

for i, sent in enumerate(test_sents):
    stand_tokens = sent.split()
    gold_token_col.append(stand_tokens)
    # if sent != '':
    #     sent = sent[0].lower() + sent[1:]
    # gold_id_col.append(tokenizer.encode(sent))

    model_tokens = tokenizer.tokenize(sent)
    model_token_col.append(model_tokens)

    model_ids = tokenizer.convert_tokens_to_ids(model_tokens)
    gold_id_col.append(model_ids)
    
    if i < 5:
        print(sent)
        print(len(stand_tokens), stand_tokens)
        print(len(model_ids), model_ids)
        ids = tokenizer.encode(sent)
        print(len(ids), ids)
        print(len(model_tokens), model_tokens)

       

My 10 year old Fuji film camera , which was a piece of crap , was much better than that . 
21 ['My', '10', 'year', 'old', 'Fuji', 'film', 'camera', ',', 'which', 'was', 'a', 'piece', 'of', 'crap', ',', 'was', 'much', 'better', 'than', 'that', '.']
25 [499, 335, 215, 625, 25147, 814, 1861, 3, 6, 84, 47, 3, 9, 1466, 13, 17081, 3, 6, 47, 231, 394, 145, 24, 3, 5]
26 [499, 335, 215, 625, 25147, 814, 1861, 3, 6, 84, 47, 3, 9, 1466, 13, 17081, 3, 6, 47, 231, 394, 145, 24, 3, 5, 1]
25 ['▁My', '▁10', '▁year', '▁old', '▁Fuji', '▁film', '▁camera', '▁', ',', '▁which', '▁was', '▁', 'a', '▁piece', '▁of', '▁crap', '▁', ',', '▁was', '▁much', '▁better', '▁than', '▁that', '▁', '.']
After taking close to 300 and the battery meter is showing all the bars on my XT ! 
18 ['After', 'taking', 'close', 'to', '300', 'and', 'the', 'battery', 'meter', 'is', 'showing', 'all', 'the', 'bars', 'on', 'my', 'XT', '!']
22 [621, 838, 885, 12, 3147, 11, 8, 3322, 3, 4401, 19, 2924, 66, 8, 6448, 30, 82, 3, 4, 382, 3, 55]
23

In [282]:
cnt = 0
index_mapping_cols = []
final_labs = []
POLARITY_DICT = ['Better', 'Different', 'Equal', 'Worse']

for index, label in enumerate(label_tuples):
    # gold_label = remove_accents(test_sents[index].lower())
    label_index = []
    for tup in label:
        map_index = init_tuple(elem_dict=elem_dict)
        for key in elem_dict:
            if tup[key] and key !='label':
                
                new_tup = " ".join(tup[key])
                encode_tup = tokenizer.encode(new_tup)
                encode_tup = encode_tup[:len(encode_tup)-1]

                if len(encode_tup):
                    # encode_tup = encode_tup[0]
                    # pos = [i for i, j in enumerate(gold_id_col[index]) if j == encode_tup]
                    start, idx = check_sublist_in_list(gold_id_col[index], encode_tup)
                    
                    if idx == -1:
                        upper_tup = new_tup[0].upper() + new_tup[1:]
                        encode_tup = tokenizer.encode(upper_tup)
                        encode_tup = encode_tup[:len(encode_tup) - 1]
                        start, idx = check_sublist_in_list(gold_id_col[index], encode_tup)
                        
                        if idx == -1:
                            lower_tup = new_tup[0].lower() + new_tup[1:]
                            encode_tup = tokenizer.encode(lower_tup)
                            encode_tup = encode_tup[:len(encode_tup) - 1]
                            start, idx = check_sublist_in_list(gold_id_col[index], encode_tup)

                            if idx == -1:
                                cnt += 1
                                print(gold_token_col[index])
                                # print(gold_id_col[index])
                                print(tup[key])
                                # print(encode_tup)
                                continue
                    if start != -1 and idx != -1:
                        map_index[key].extend([start, idx])
            else:
                if tup[key]:
                    for polar in POLARITY_DICT:
                        if polar in tup[key]:
                            map_index[key] = polar
                            break
                else:
                    map_index[key] = tup[key]
                    
                final_labs.append(map_index[key])
        label_index.append(map_index)
    
    index_mapping_cols.append(label_index)
    
print(cnt)

0


In [283]:
labs = []
for final in final_labs:
    if final:
        labs.append(final)

print(set(labs))

{'Different', 'Equal', 'Better', 'Worse'}


In [284]:
index_mapping_cols

[[],
 [{'subject': [], 'object': [], 'aspect': [], 'predicate': [], 'label': ''}],
 [{'subject': [4, 14],
   'object': [4, 7],
   'aspect': [2, 36],
   'predicate': [1, 35],
   'label': 'Better'}],
 [{'subject': [1, 6],
   'object': [3, 19],
   'aspect': [3, 14],
   'predicate': [10, 7],
   'label': 'Better'},
  {'subject': [1, 6],
   'object': [3, 19],
   'aspect': [],
   'predicate': [6, 25],
   'label': 'Better'}],
 [{'subject': [],
   'object': [],
   'aspect': [1, 3],
   'predicate': [1, 5],
   'label': 'Better'}],
 [{'subject': [], 'object': [], 'aspect': [], 'predicate': [], 'label': ''}],
 [],
 [{'subject': [], 'object': [], 'aspect': [], 'predicate': [], 'label': ''}],
 [{'subject': [], 'object': [], 'aspect': [], 'predicate': [], 'label': ''}],
 [{'subject': [], 'object': [], 'aspect': [], 'predicate': [], 'label': ''}],
 [{'subject': [], 'object': [], 'aspect': [], 'predicate': [], 'label': ''}],
 [{'subject': [3, 4],
   'object': [],
   'aspect': [2, 19],
   'predicate': [1

In [285]:
def token_mapping_bert(tokenizer, model_token_col, model_id_col, gold_token_col):
    """
    :param model_token_col: a list of token list by BertTokenizer (with [cls] and [sep])
    :param gold_token_col: a list char list
    :return: a map: {standard_token_index: [model_indexes]}
    """
    assert len(model_token_col) == len(gold_token_col), "bert data length not equal to char data length"

    mapping_col = []
    for index, model_tokens in enumerate(model_token_col):
        seq_map, model_index, token_index = {}, 0, 0 ## bert token except [CLS] in index 0
        seq_model_token, seq_model_id, seq_gold_token = model_tokens, model_id_col[index], gold_token_col[index]

        # print(seq_model_token)
        # print(seq_gold_token)
        
        while model_index < len(seq_model_token) and token_index < len(seq_gold_token):
            seq_map[token_index] = [model_index]

            if seq_model_token[model_index] == '<unk>' or '...' in seq_model_token[model_index]:
                model_index = model_index + 1
                token_index = token_index + 1
                continue

            token_length = len(seq_gold_token[token_index])
            model_length = len(seq_model_token[model_index])

            # print(token_length, model_length, seq_gold_token[token_index], seq_model_token[model_index])

            if seq_model_token[model_index].find("▁") != -1:
                model_length = len(seq_model_token[model_index]) - 1

            ##Tìm tất cả các subtokens bị tách ra
            while token_length > model_length:
                model_index = model_index + 1
                seq_map[token_index].append(model_index)

                if seq_model_token[model_index] != '<unk>':
                    model_length = model_length + len(seq_model_token[model_index])
                else:
                    model_length += 1
                    # token_index += 1
                    # model_index += 1

                # if seq_model_token[model_index].find("_") != -1:
                #     model_length -= 1
            
            
            assert model_length == token_length, "appear mapping error"

            token_index += 1
            model_index += 1
        
        seq_map[token_index] = [model_index]
        # print(seq_map)
        mapping_col.append(seq_map)
        # print(mapping_col)
    return mapping_col

mapping_col = token_mapping_bert(tokenizer, model_token_col, gold_id_col, gold_token_col)
mapping_col

# # print key with val 100
# position = val_list.index(100)
# print(key_list[position])


[{0: [0],
  1: [1],
  2: [2],
  3: [3],
  4: [4],
  5: [5],
  6: [6],
  7: [7, 8],
  8: [9],
  9: [10],
  10: [11, 12],
  11: [13],
  12: [14],
  13: [15],
  14: [16, 17],
  15: [18],
  16: [19],
  17: [20],
  18: [21],
  19: [22],
  20: [23, 24],
  21: [25]},
 {0: [0],
  1: [1],
  2: [2],
  3: [3],
  4: [4],
  5: [5],
  6: [6],
  7: [7],
  8: [8, 9],
  9: [10],
  10: [11],
  11: [12],
  12: [13],
  13: [14],
  14: [15],
  15: [16],
  16: [17, 18, 19],
  17: [20, 21],
  18: [22]},
 {0: [0],
  1: [1],
  2: [2, 3, 4],
  3: [5],
  4: [6],
  5: [7],
  6: [8, 9],
  7: [10],
  8: [11, 12],
  9: [13],
  10: [14, 15],
  11: [16, 17],
  12: [18],
  13: [19, 20, 21, 22],
  14: [23],
  15: [24, 25],
  16: [26],
  17: [27, 28],
  18: [29, 30],
  19: [31],
  20: [32],
  21: [33, 34],
  22: [35],
  23: [36],
  24: [37],
  25: [38],
  26: [39],
  27: [40],
  28: [41, 42],
  29: [43]},
 {0: [0, 1, 2, 3, 4],
  1: [5],
  2: [6],
  3: [7],
  4: [8],
  5: [9, 10, 11],
  6: [12, 13],
  7: [14, 15, 16],
  8

In [286]:
def corresponding_key(val, dictionary):
    for k, v in dictionary.items():
        if val in v:
            return k
        
def check_num_tuple(quintupe):
    cnt = 0
    for key in quintupe:
        if quintupe[key]:
            cnt +=1
    return cnt

print(check_num_tuple({'subject': ['8&&Samsung', '9&&Galaxy', '10&&Z', '11&&Fold3'], 'object': [], 'aspect': [], 'predicate': [], 'label': 'COM+'}))

2


In [287]:
formated_label = []
for index in range(len(test_sents)):
    tup_col = []
    for map_tup in index_mapping_cols[index]:
        if map_tup == null_label:
            continue
        else:
            # print(map_tup)
            gold_tokens = gold_token_col[index]

            label = init_tuple(elem_dict)
            
            for key in elem_dict:
                if map_tup[key] and key != 'label':
                   
                    length, idx = map_tup[key]
                    # print(length, idx)
                    s_index = corresponding_key(idx, mapping_col[index])
                    e_index = corresponding_key(idx+length, mapping_col[index])

                    label[key] = gold_tokens[s_index:e_index]
                    
                    for j in range(len(label[key])):
                        label[key][j] = f"{s_index+j+1}&&{label[key][j]}"
                    
                else:
                    label[key] = map_tup[key]
        
        if check_num_tuple(label) < 3:
            print(test_sents[index])
            print(label)
        elif len(label['subject']) == 0 and len(label['object']) == 0:
            print(test_sents[index])
            print(label)

        tup_col.append(label)

        
    formated_label.append(tup_col)

formated_label         
                
         

Outdoors the clarity is outstanding . 
{'subject': [], 'object': [], 'aspect': ['3&&clarity'], 'predicate': ['5&&outstanding'], 'label': 'Better'}
And of course the build quality is on a completely different level . 
{'subject': [], 'object': [], 'aspect': ['5&&build', '6&&quality'], 'predicate': ['11&&different'], 'label': 'Different'}
Memory and Storage The SD800 IS uses SD ( Secure Digital ) , MMC ( MultiMedia Card ) , and SDHC ( High capacity SD ) cards for storage . 
{'subject': ['5&&SD800', '6&&IS'], 'object': [], 'aspect': [], 'predicate': [], 'label': ''}
However , focus accuracy was not as impressive . 
{'subject': [], 'object': [], 'aspect': ['3&&focus', '4&&accuracy'], 'predicate': ['6&&not', '7&&as', '8&&impressive'], 'label': 'Worse'}
Realistically there will be no difference in your photos . 
{'subject': [], 'object': [], 'aspect': ['9&&photos'], 'predicate': ['5&&no', '6&&difference'], 'label': 'Equal'}
The less apparent difference is performance . 
{'subject': [], 'obje

[[],
 [],
 [{'subject': ['11&&Powershot', '12&&SD630'],
   'object': ['6&&Canon', '7&&Powershot', '8&&cameras'],
   'aspect': ['24&&LCD', '25&&monitor'],
   'predicate': ['23&&larger'],
   'label': 'Better'}],
 [{'subject': ['3&&LCD'],
   'object': ['11&&optical', '12&&viewfinder'],
   'aspect': ['8&&framing'],
   'predicate': ['4&&gives', '5&&you', '6&&99-100', '7&&%', '8&&framing'],
   'label': 'Better'},
  {'subject': ['3&&LCD'],
   'object': ['11&&optical', '12&&viewfinder'],
   'aspect': [],
   'predicate': ['16&&85', '17&&%', '18&&framing'],
   'label': 'Better'}],
 [{'subject': [],
   'object': [],
   'aspect': ['3&&clarity'],
   'predicate': ['5&&outstanding'],
   'label': 'Better'}],
 [],
 [],
 [],
 [],
 [],
 [],
 [{'subject': ['4&&both', '5&&Canons'],
   'object': [],
   'aspect': ['16&&AF'],
   'predicate': ['18&&better'],
   'label': 'Better'},
  {'subject': ['4&&both', '5&&Canons'],
   'object': [],
   'aspect': ['19&&white', '20&&balance', '21&&adjustments'],
   'predicat

In [288]:
for index, sent in enumerate(test_sents):
    print(sent)
    print(formated_label[index])
    print('='*20)

My 10 year old Fuji film camera , which was a piece of crap , was much better than that . 
[]
After taking close to 300 and the battery meter is showing all the bars on my XT ! 
[]
As for comparisions with other Canon Powershot cameras , the Powershot SD630 does n't have a view finder , but has a larger LCD monitor in its place . 
[{'subject': ['11&&Powershot', '12&&SD630'], 'object': ['6&&Canon', '7&&Powershot', '8&&cameras'], 'aspect': ['24&&LCD', '25&&monitor'], 'predicate': ['23&&larger'], 'label': 'Better'}]
Viewfinder/LCD The LCD gives you 99-100 % framing while the optical viewfinder gives you about 85 % framing . 
[{'subject': ['3&&LCD'], 'object': ['11&&optical', '12&&viewfinder'], 'aspect': ['8&&framing'], 'predicate': ['4&&gives', '5&&you', '6&&99-100', '7&&%', '8&&framing'], 'label': 'Better'}, {'subject': ['3&&LCD'], 'object': ['11&&optical', '12&&viewfinder'], 'aspect': [], 'predicate': ['16&&85', '17&&%', '18&&framing'], 'label': 'Better'}]
Outdoors the clarity is outsta

In [289]:
# -1: worse, 0: similar, 1: better, 2: different
label2id = {
    "Better": 1,
    "Worse": -1,
    'Equal': 0,
    "Different": 2
}

null_pos_label  = (-1, -1, -1, -1, -1, -1, -1, -1, -1)
def extract_start_end_predict_position(arr):
    eles = []
    for item in arr:
        ele = item.split("&&")[0]
        eles.append(int(ele))

    if len(eles) != 0:
        return eles[0], eles[-1]

    return -1, -1

def get_element_position_each_predict(label):
    pos_tuple = []
    pos_tuple.append(label2id[label['label']])
    for key in label:
        if key != 'label':
            s_index, e_index = extract_start_end_predict_position(label[key])
            pos_tuple.extend([s_index, e_index])    
    return tuple(pos_tuple)

test_dict = {'subject': [], 'object': ['6&&digital', '7&&Elph', '8&&line'], 'aspect': [], 'predicate': ['1&&Like'], 'label': 'Equal'}
print(get_element_position_each_predict(test_dict)) 

predict_dicts = formated_label
def parse_element_position_predicts(predict_dicts):
    predict_pos_list = []
    for labels in predict_dicts:
        pos_list = []
        for _l in labels:
            if _l['label'] != '':
                pos_tuple = get_element_position_each_predict(_l)
                pos_list.append(pos_tuple)
            else:
                print(_l)

        predict_pos_list.append(pos_list)

    return predict_pos_list

test_predict_pos_tuples = parse_element_position_predicts(predict_dicts)
    


(0, -1, -1, 6, 8, -1, -1, 1, 1)
{'subject': ['5&&SD800', '6&&IS'], 'object': [], 'aspect': [], 'predicate': [], 'label': ''}
{'subject': ['4&&Nikon', '5&&cameras'], 'object': [], 'aspect': [], 'predicate': [], 'label': ''}
{'subject': ['6&&Canon', '7&&EOS', '8&&Digital', '9&&Rebel'], 'object': [], 'aspect': [], 'predicate': [], 'label': ''}
{'subject': ['3&&M1', '4&&resolution'], 'object': [], 'aspect': [], 'predicate': [], 'label': ''}
{'subject': ['3&&This', '4&&camera'], 'object': [], 'aspect': [], 'predicate': [], 'label': ''}
{'subject': ['2&&camera'], 'object': [], 'aspect': [], 'predicate': [], 'label': ''}
{'subject': ['2&&Nikon', '3&&D70'], 'object': [], 'aspect': [], 'predicate': [], 'label': ''}
{'subject': ['7&&these', '8&&cameras'], 'object': [], 'aspect': [], 'predicate': [], 'label': ''}
{'subject': ['1&&Canon', "2&&'s", '3&&CMOS', '4&&sensor'], 'object': [], 'aspect': [], 'predicate': [], 'label': ''}
{'subject': ['4&&SD', '5&&slot'], 'object': [], 'aspect': [], 'predic

In [290]:
test_truths[:5]

[['[[5&&Fuji 6&&film 7&&camera];[20&&that];[];[18&&better];[1]]'],
 ['[[];[];[];[];[]]'],
 ['[[5&&other 6&&Canon 7&&Powershot 8&&cameras];[11&&Powershot 12&&SD630];[24&&LCD 25&&monitor];[23&&larger];[1]]',
  "[[5&&other 6&&Canon 7&&Powershot 8&&cameras];[11&&Powershot 12&&SD630];[17&&view 18&&finder];[13&&does 14&&n't 15&&have];[-1]]"],
 ['[[];[];[];[];[]]'],
 ['[[];[];[];[];[]]']]

In [291]:
def parse_each_english_label_string(text):
    """
    Parses a string into a list of tuples with the desired format.
    """
    label_tuple = []

    text = text[2:-2]
    print(text)
    string_split = text.split("];[")
    print(string_split)
    string_split = [string.split(" ") for string in string_split]

    if '-1' in string_split[-1][0]:
        label_tuple.extend([-1])
    else:
        label_tuple.append(int(string_split[-1][0][0]))
        
    

    for _list in string_split[:-1]:
        element_list = [int(item.split("&&")[0]) if "&&" in item else -1 for item in _list]
        #  element = " ".join(element_list)
        
        if len(element_list) == 0:
            s_index, e_index = -1, -1
        elif len(element_list) == 1:
            s_index, e_index = element_list[0], element_list[0]
        else:
            s_index, e_index = element_list[0], element_list[-1]
        label_tuple.extend([s_index, e_index])
    # label_tuple = [string.split("&&")[-1] if "&&" in string else "[UNK]" for string in string_split[:-1]]
    
    
    return tuple(label_tuple)

string = "[[6&&this];[19&&SLR];[13&&image 14&&quality];[12&&best];[1]]"
parsed_tuples = parse_each_english_label_string(string)

print(parsed_tuples)  

6&&this];[19&&SLR];[13&&image 14&&quality];[12&&best];[1
['6&&this', '19&&SLR', '13&&image 14&&quality', '12&&best', '1']
(1, 6, 6, 19, 19, 13, 14, 12, 12)


In [292]:
def parse_eng_labels(label_list):
    eng_null_label = '[[];[];[];[];[]]'
    eng_null_tuple = ('[UNK]', '[UNK]', '[UNK]', '[UNK]','[UNK]')
    comp_label_list = []
    for labels in label_list:
        comp_labels = []
        for _label in labels:
            if _label != eng_null_label:
                comp_labels.append(parse_each_english_label_string(_label))
            # else:
            #     comp_labels.append([])
        comp_label_list.append(comp_labels)
    return comp_label_list

test_truth_pos_tuples = parsed_label_list = parse_eng_labels(test_truths)

5&&Fuji 6&&film 7&&camera];[20&&that];[];[18&&better];[1
['5&&Fuji 6&&film 7&&camera', '20&&that', '', '18&&better', '1']
5&&other 6&&Canon 7&&Powershot 8&&cameras];[11&&Powershot 12&&SD630];[24&&LCD 25&&monitor];[23&&larger];[1
['5&&other 6&&Canon 7&&Powershot 8&&cameras', '11&&Powershot 12&&SD630', '24&&LCD 25&&monitor', '23&&larger', '1']
5&&other 6&&Canon 7&&Powershot 8&&cameras];[11&&Powershot 12&&SD630];[17&&view 18&&finder];[13&&does 14&&n't 15&&have];[-1
['5&&other 6&&Canon 7&&Powershot 8&&cameras', '11&&Powershot 12&&SD630', '17&&view 18&&finder', "13&&does 14&&n't 15&&have", '-1']
2&&digital 3&&DSRL];[];[];[1&&Better];[1
['2&&digital 3&&DSRL', '', '', '1&&Better', '1']
];[6&&digital 7&&Elph];[14&&manual 15&&controls];[1&&Like];[0
['', '6&&digital 7&&Elph', '14&&manual 15&&controls', '1&&Like', '0']
4&&both 5&&Canons];[4&&both 5&&Canons];[16&&AF];[15&&faster];[1
['4&&both 5&&Canons', '4&&both 5&&Canons', '16&&AF', '15&&faster', '1']
4&&both 5&&Canons];[4&&both 5&&Canons];[28&&

In [293]:
test_truth_pos_tuples

[[(1, 5, 7, 20, 20, -1, -1, 18, 18)],
 [],
 [(1, 5, 8, 11, 12, 24, 25, 23, 23), (-1, 5, 8, 11, 12, 17, 18, 13, 15)],
 [],
 [],
 [(1, 2, 3, -1, -1, -1, -1, 1, 1)],
 [(0, -1, -1, 6, 7, 14, 15, 1, 1)],
 [],
 [],
 [],
 [],
 [(1, 4, 5, 4, 5, 16, 16, 15, 15),
  (1, 4, 5, 4, 5, 28, 28, 27, 27),
  (1, 4, 5, 4, 5, 10, 10, 9, 9),
  (1, 4, 5, 4, 5, 19, 21, 18, 18),
  (1, 4, 5, 4, 5, 24, 25, 23, 23)],
 [],
 [(2, -1, -1, -1, -1, 5, 6, 11, 11)],
 [],
 [],
 [(-1, -1, -1, -1, -1, 3, 4, 6, 8)],
 [],
 [],
 [],
 [(1, -1, -1, 9, 10, 3, 3, 15, 15), (1, -1, -1, 9, 10, 1, 1, 6, 6)],
 [(1, 11, 12, 5, 6, -1, -1, 8, 8)],
 [(1, 4, 4, 15, 16, 13, 13, 7, 7)],
 [],
 [(1, 2, 2, 18, 18, 8, 8, 9, 9),
  (1, 2, 2, 18, 18, 4, 5, 6, 6),
  (1, 2, 2, 18, 18, 12, 14, 15, 15)],
 [],
 [(1, 10, 10, -1, -1, -1, -1, 12, 13),
  (1, 1, 1, -1, -1, 7, 7, 5, 6),
  (1, 37, 38, -1, -1, -1, -1, 41, 41)],
 [(1, 23, 25, 52, 54, 38, 38, 36, 36),
  (1, 11, 13, 41, 41, 38, 38, 36, 36),
  (1, 23, 25, 52, 54, 38, 38, 36, 36),
  (1, 11, 13, 41, 

In [294]:
def exact_metric(pred, gold):
    assert len(pred) == len(gold)
    gold_num = 0
    rel_num = 0
    ent_num = 0
    right_num = 0
    pred_num = 0
    for sent_idx, prediction in enumerate(pred):
        gold_num += len(gold[sent_idx])
        pred_correct_num = 0
        pred_num += len(prediction)

        # if gold_num == 0 and pred_num==0:
        #     gold_num += 1
        #     pred_num += 1
        #     right_num += 1
    
        for ele in prediction:
           
            if ele in gold[sent_idx]:  # eg: (3, 3, 5, 32, 34, 24, 27, 28, 29)
                # print(ele)
                # print(gold[sent_idx])
                right_num += 1
                pred_correct_num += 1
            if ele[0] in [e[0] for e in gold[sent_idx]]: # compute the correct rel number
                rel_num += 1
            if ele[1:] in [e[1:] for e in gold[sent_idx]]: # computer the correct four elements
                ent_num += 1

    if pred_num == 0:
        precision = -1
        r_p = -1
        e_p = -1
    else:
        precision = (right_num + 0.0) / pred_num
        e_p = (ent_num + 0.0) / pred_num
        r_p = (rel_num + 0.0) / pred_num

    if gold_num == 0:
        recall = -1
        r_r = -1
        e_r = -1
    else:
        recall = (right_num + 0.0) / gold_num
        e_r = ent_num / gold_num
        r_r = rel_num / gold_num

    if (precision == -1) or (recall == -1) or (precision + recall) <= 0.:
        f_measure = -1
    else:
        f_measure = 2 * precision * recall / (precision + recall)

    if (e_p == -1) or (e_r == -1) or (e_p + e_r) <= 0.:
        e_f = -1
    else:
        e_f = 2 * e_r * e_p / (e_p + e_r)

    if (r_p == -1) or (r_r == -1) or (r_p + r_r) <= 0.:
        r_f = -1
    else:
        r_f = 2 * r_p * r_r / (r_r + r_p)

    precision = precision * 100
    recall = recall * 100
    f_measure = f_measure * 100

    print("gold_num = ", gold_num, " pred_num = ", pred_num, " right_num = ", right_num, " relation_right_num = ", rel_num, " entity_right_num = ", ent_num)
    print("cee-precision = ", e_p, " cee-recall = ", e_r, "cee-f1_value = ", e_f)
    print("cpc-precision = ", r_p, " cpc-recall = ", r_r, "cpc-f1_value = ", r_f)
    print("q5-precision = ", precision, " q5-recall = ", recall, " q5-f1_value = ", f_measure)
    return {"precision": precision, "recall": recall, "f1": f_measure}

exact_metric(test_predict_pos_tuples, test_truth_pos_tuples)

gold_num =  492  pred_num =  420  right_num =  91  relation_right_num =  295  entity_right_num =  106
cee-precision =  0.2523809523809524  cee-recall =  0.21544715447154472 cee-f1_value =  0.2324561403508772
cpc-precision =  0.7023809523809523  cpc-recall =  0.5995934959349594 cpc-f1_value =  0.6469298245614035
q5-precision =  21.666666666666668  q5-recall =  18.495934959349594  q5-f1_value =  19.956140350877195


{'precision': 21.666666666666668,
 'recall': 18.495934959349594,
 'f1': 19.956140350877195}

In [295]:
def tuple_to_five_ele(ele_tuple):
    ele_list = list(ele_tuple)
    rel_pred, sub_pred, obj_pred, aspect_pred, opinion_pred = ele_list[0], (ele_list[1],ele_list[2]),(ele_list[3],ele_list[4]), \
    (ele_list[5],ele_list[6]),(ele_list[7],ele_list[8])
    return rel_pred, sub_pred, obj_pred, aspect_pred, opinion_pred

def convert_tuple_to_set(tuple1):
    rel_set = set()
    for i in range(tuple1[0], tuple1[1]+1): # 左闭右闭,闭需要+1
        rel_set.add(i)
    return rel_set

def binary_metric(pred, gold):
    assert len(pred) == len(gold)
    gold_num = 0
    rel_num = 0
    ent_num = 0
    right_num = 0
    pred_num = 0
    for sent_idx, prediction in enumerate(pred):
        gold_num += len(gold[sent_idx])
        pred_correct_num = 0
        pred_num += len(prediction)

        for ele_pred in prediction:
            for ele_gold in gold[sent_idx]:
                # ele_pred = (3, 3, 5, 32, 34, 24, 27, 28, 29)
                # ele_gold = (3, 3, 4, 32, 34, 24, 26, 28, 29)
                rel_pred, sub_pred, obj_pred, aspect_pred, opinion_pred = tuple_to_five_ele(ele_pred)
                rel_gold, sub_gold, obj_gold, aspect_gold, opinion_gold = tuple_to_five_ele(ele_gold)

                sub_pred_set, obj_pred_set, asp_pred_set, op_pred_set = convert_tuple_to_set(sub_pred), convert_tuple_to_set(obj_pred), convert_tuple_to_set(aspect_pred), convert_tuple_to_set(opinion_pred)
                sub_gold_set, obj_gold_set, asp_gold_set, op_gold_set = convert_tuple_to_set(sub_gold), convert_tuple_to_set(obj_gold), convert_tuple_to_set(aspect_gold), convert_tuple_to_set(opinion_gold)

                if (rel_pred==rel_gold) and (sub_pred_set&sub_gold_set) and (obj_pred_set & obj_gold_set) and (asp_pred_set & asp_gold_set) and (op_pred_set & op_gold_set):
                    right_num += 1
                
                if rel_pred==rel_gold:
                    rel_num +=1
                
                if (sub_pred_set&sub_gold_set) and (obj_pred_set & obj_gold_set) and (asp_pred_set & asp_gold_set) and (op_pred_set & op_gold_set):
                    ent_num += 1

    if pred_num == 0:
        precision = -1
        r_p = -1
        e_p = -1
    else:
        precision = (right_num + 0.0) / pred_num
        e_p = (ent_num + 0.0) / pred_num
        r_p = (rel_num + 0.0) / pred_num

    if gold_num == 0:
        recall = -1
        r_r = -1
        e_r = -1
    else:
        recall = (right_num + 0.0) / gold_num
        e_r = ent_num / gold_num
        r_r = rel_num / gold_num

    if (precision == -1) or (recall == -1) or (precision + recall) <= 0.:
        f_measure = -1
    else:
        f_measure = 2 * precision * recall / (precision + recall)

    precision = precision * 100
    recall = recall * 100
    f_measure = f_measure * 100

    print("+++++++++++++Binary Results ++++++++++++++++++++++++++==")
    print("gold_num = ", gold_num, " pred_num = ", pred_num, " right_num = ", right_num, " relation_right_num = ", rel_num, " entity_right_num = ", ent_num)
    print("precision = ", precision, " recall = ", recall, " f1_value = ", f_measure)
    return {"Binary precision": precision, " Binary  recall": recall, "Binary  f1": f_measure}

def proportional_metric(pred, gold):
    assert len(pred) == len(gold)
    gold_num = 0
    rel_num = 0
    ent_num = 0
    right_num = 0
    pred_num = 0
    for sent_idx, prediction in enumerate(pred):
        gold_num += len(gold[sent_idx])
        pred_correct_num = 0
        
        pred_num += len(prediction)

        for ele_pred in prediction:
            for ele_gold in gold[sent_idx]:
                # ele_pred = (3, 3, 5, 32, 34, 24, 27, 28, 29)
                # ele_gold = (3, 3, 4, 32, 34, 24, 26, 28, 29)
                rel_pred, sub_pred, obj_pred, aspect_pred, opinion_pred = tuple_to_five_ele(ele_pred)
                rel_gold, sub_gold, obj_gold, aspect_gold, opinion_gold = tuple_to_five_ele(ele_gold)

                sub_pred_set, obj_pred_set, asp_pred_set, op_pred_set = convert_tuple_to_set(sub_pred), convert_tuple_to_set(obj_pred), convert_tuple_to_set(aspect_pred), convert_tuple_to_set(opinion_pred)
                sub_gold_set, obj_gold_set, asp_gold_set, op_gold_set = convert_tuple_to_set(sub_gold), convert_tuple_to_set(obj_gold), convert_tuple_to_set(aspect_gold), convert_tuple_to_set(opinion_gold)

                if (rel_pred==rel_gold) and (sub_pred_set&sub_gold_set) and (obj_pred_set & obj_gold_set) and (asp_pred_set & asp_gold_set) and (op_pred_set & op_gold_set):
                    sub_union = sub_pred_set & sub_gold_set
                    obj_union = obj_pred_set & obj_gold_set
                    asp_union = asp_pred_set & asp_gold_set
                    op_union = op_pred_set & op_gold_set
                    all_union_len = len(sub_union) + len(obj_union) + len(asp_union) + len(op_union)
                    all_gold_len = len(sub_gold_set) + len(obj_gold_set) + len(asp_gold_set) + len(op_gold_set)
                    cur_num = all_union_len / all_gold_len
                    right_num = right_num + cur_num
                
                if rel_pred==rel_gold:
                    rel_num +=1
                
                if (sub_pred_set&sub_gold_set) and (obj_pred_set & obj_gold_set) and (asp_pred_set & asp_gold_set) and (op_pred_set & op_gold_set):
                    ent_num += 1

    if pred_num == 0:
        precision = -1
        r_p = -1
        e_p = -1
    else:
        precision = (right_num + 0.0) / pred_num
        e_p = (ent_num + 0.0) / pred_num
        r_p = (rel_num + 0.0) / pred_num

    if gold_num == 0:
        recall = -1
        r_r = -1
        e_r = -1
    else:
        recall = (right_num + 0.0) / gold_num
        e_r = ent_num / gold_num
        r_r = rel_num / gold_num

    if (precision == -1) or (recall == -1) or (precision + recall) <= 0.:
        f_measure = -1
    else:
        f_measure = 2 * precision * recall / (precision + recall)
        
    precision = precision * 100
    recall = recall * 100
    f_measure = f_measure * 100

    print("+++++++++++++ Proportional Results ++++++++++++++++++++++++++==")
    print("gold_num = ", gold_num, " pred_num = ", pred_num, " right_num = ", right_num, " relation_right_num = ", rel_num, " entity_right_num = ", ent_num)
    print("precision = ", precision, " recall = ", recall, " f1_value = ", f_measure)
    return {"Proportional precision": precision, " Proportional  recall": recall, "Proportional  f1": f_measure}

binary_metric(test_predict_pos_tuples, test_truth_pos_tuples)
proportional_metric(test_predict_pos_tuples, test_truth_pos_tuples)


+++++++++++++Binary Results ++++++++++++++++++++++++++==
gold_num =  492  pred_num =  420  right_num =  144  relation_right_num =  437  entity_right_num =  169
precision =  34.285714285714285  recall =  29.268292682926827  f1_value =  31.578947368421055
+++++++++++++ Proportional Results ++++++++++++++++++++++++++==
gold_num =  492  pred_num =  420  right_num =  133.53439893439892  relation_right_num =  437  entity_right_num =  169
precision =  31.793904508190217  recall =  27.141137994796527  f1_value =  29.283859415438357


{'Proportional precision': 31.793904508190217,
 ' Proportional  recall': 27.141137994796527,
 'Proportional  f1': 29.283859415438357}